## Quickstart

This notebook mirrors the [Cecil quickstart](https://docs.cecil.earth/quickstart) end-to-end. By the time you reach the bottom you'll have:

- a Cecil API key set up in this environment,
- an Area of Interest (AOI),
- a subscription to a dataset on that AOI,
- an `xarray.Dataset` you can analyse and plot.

If you haven't already, [sign up at cecil.earth](https://cecil.earth) before running this notebook.

### 1. Install the SDK

Install the [`cecil`](https://pypi.org/project/cecil/) Python package. On Colab the `%pip` magic installs into the active kernel.

In [ ]:
%pip install -q cecil

### 2. Authenticate

Grab an API key from your account at [cecil.earth](https://cecil.earth) (or run `cecil login` locally). The SDK reads it from the `CECIL_API_KEY` environment variable.

On Colab the cell below will prompt you to paste the key — it stays in memory for the session and isn't persisted.

In [ ]:
import getpass
import os

if not os.environ.get("CECIL_API_KEY"):
    os.environ["CECIL_API_KEY"] = getpass.getpass("Cecil API key: ")

In [ ]:
from cecil import Client

client = Client()

### 3. Create an Area of Interest

An AOI is a reusable polygon you can subscribe many datasets to. The starter polygon below is a small (~1100 ha) box in the Brazilian Amazon — swap in your own GeoJSON when you're ready.

`external_ref` is an optional string you can use to tag the AOI with an identifier from your own system.

In [ ]:
aoi = client.create_aoi(
    geometry={
        "type": "Polygon",
        "coordinates": [[
            [-56.00, -10.00],
            [-55.90, -10.00],
            [-55.90, -9.90],
            [-56.00, -9.90],
            [-56.00, -10.00],
        ]],
    },
    external_ref="quickstart-sample-amazon",
)
aoi

### 4. Subscribe to a dataset

Browse [docs.cecil.earth/datasets](https://docs.cecil.earth/datasets) to find a dataset ID. The cell below uses WorldPop population counts at 1 km resolution as an example — substitute any dataset ID you have access to.

Subscriptions are processed asynchronously: a few moments after you create one, the data is staged in object storage and ready to load.

In [ ]:
subscription = client.create_subscription(
    aoi_id=aoi.id,
    dataset_id="0273552c-b519-4ff8-906e-f4969c3a4fc7",  # WorldPop population counts, 1 km
)
subscription

In [ ]:
import time

from cecil.errors import HTTPError

# The subscription's data files appear in object storage once processing finishes.
# Until then, requesting them returns an HTTP error — we retry until it succeeds.
while True:
    try:
        client.load_xarray(subscription.id)
        print("  status: ready")
        break
    except HTTPError as err:
        print(f"  status: not ready yet ({err.status_code}) — waiting…")
        time.sleep(10)

### 5. Load and analyse

For raster datasets, `client.load_xarray(...)` returns an [`xarray.Dataset`](https://docs.xarray.dev/en/stable/generated/xarray.Dataset.html) backed by lazy dask arrays. For vector datasets, use `client.load_dataframe(...)` to get a `pandas.DataFrame` instead.

In [ ]:
ds = client.load_xarray(subscription.id)
ds

In [ ]:
import matplotlib.pyplot as plt

var_name = list(ds.data_vars)[0]
layer = ds[var_name]
if "time" in layer.dims:
    layer = layer.isel(time=-1)

layer.plot(cmap="magma")
plt.title(var_name)
plt.show()

### Next steps

- [Browse the dataset catalogue](https://docs.cecil.earth/datasets)
- [More worked examples](https://docs.cecil.earth/examples)
- [Full SDK reference](https://docs.cecil.earth/sdk)